In [1]:
!pip install chromadb sentence-transformers

In [2]:
import chromadb
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer

c:\Users\DELL\Desktop\RAG focused corpus\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1895.46it/s]


In [5]:
from pathlib import Path

import chromadb

candidate_paths = [
    Path.cwd() / "data" / "chroma_db",
    Path.cwd().parent / "data" / "chroma_db",
    Path.cwd().parent.parent / "data" / "chroma_db",
]

chroma_path = next((path for path in candidate_paths if path.exists()), candidate_paths[0])
client = chromadb.PersistentClient(path=str(chroma_path))
collection_name = "studybuddy_rag"
collection = client.get_or_create_collection(name=collection_name)

print(f"Connected to ChromaDB at: {chroma_path}")
print(f"Collection '{collection_name}' is ready")

Connected to ChromaDB at: c:\Users\DELL\Desktop\RAG focused corpus\data\chroma_db
Collection 'studybuddy_rag' is ready


In [7]:
def retrieve_documents(query, n_results=3):
    query_embedding = embedding_model.encode(query, convert_to_tensor=False).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    documents = []
    for doc, meta, distance in zip(
        results.get("documents", [[]])[0],
        results.get("metadatas", [[]])[0],
        results.get("distances", [[]])[0],
    ):
        documents.append({
            "text": doc,
            "metadata": meta,
            "distance": distance,
        })

    return documents


query = "What is Retrieval-Augmented Generation?"
results = retrieve_documents(query)

print(results)

[]


In [9]:
for i, doc in enumerate(results):
    print("=" * 80)
    print(f"Result {i+1}")
    print(f"Source : {doc['metadata'].get('source', 'N/A')}")
    print(f"Chunk  : {doc['metadata'].get('chunk_id', 'N/A')}")
    print()
    print(doc['text'])

In [10]:
for i, doc in enumerate(results):
    print("=" * 80)
    print(f"Result {i+1}")
    print(f"Source   : {doc['metadata'].get('source', 'N/A')}")
    print(f"Chunk ID : {doc['metadata'].get('chunk_id', 'N/A')}")
    print(f"Distance : {doc['distance']:.4f}")
    print("-" * 80)
    print(doc["text"])
    print()

In [11]:
print("Documents in collection:", collection.count())

Documents in collection: 0


In [13]:
query = "What is Retrieval-Augmented Generation?"
results = retrieve_documents(query)

print(results)
print(len(results))

[]
0


In [14]:
query_embedding = embedding_model.encode(
    "What is Retrieval-Augmented Generation?",
    convert_to_tensor=False
).tolist()

raw_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(raw_results)

{'ids': [[]], 'embeddings': None, 'documents': [[]], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[]], 'distances': [[]]}


In [15]:
{
    'documents': [[]],
    'metadatas': [[]],
    'distances': [[]]
}

{'documents': [[]], 'metadatas': [[]], 'distances': [[]]}

In [16]:
print(collection.count())

0


In [17]:
print(raw_results)

{'ids': [[]], 'embeddings': None, 'documents': [[]], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[]], 'distances': [[]]}
